# WoundWatch — Gemma 4 Fine-tuning with Unsloth

**Kaggle 실행 전 체크리스트:**
1. Settings → Accelerator: **GPU T4 x2** (또는 P100)
2. Add Data → `laithjj/diabetic-foot-ulcer-dfu`
3. Add Data → `leoscode/wound-segmentation-images`
4. Secrets → `HF_TOKEN` (HuggingFace Write 권한 토큰)
5. Settings → Internet → ON (패키지 설치 필요)

**목표:** Gemma 4 4B 멀티모달을 DFU 이미지 분석 태스크에 파인튜닝  
**출력 포맷:** ai_service.py와 동일한 JSON (infection, ischemia, severity, wound_area_cm2, description, confidence)  
**저장:** GGUF (Ollama) + HuggingFace Hub

In [ ]:
# ── 1. 의존성 설치 ────────────────────────────────────────────────────────────
import subprocess, sys

subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet",
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
], check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet", "--no-deps",
    "trl>=0.9.0", "peft", "accelerate", "bitsandbytes"
], check=True)

print("설치 완료")

In [ ]:
# ── 2. 임포트 ────────────────────────────────────────────────────────────────
import json
import random
import re
from pathlib import Path

import cv2
import numpy as np
from PIL import Image
from datasets import Dataset

from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

random.seed(42)
print("임포트 완료")

In [ ]:
# ── 3. 시스템 프롬프트 (ai_service.py와 동일하게 유지) ────────────────────────
# 학습 시 이 프롬프트를 system turn에 넣어 모델이 JSON 출력을 학습하도록 함

SYSTEM_PROMPT = """You are a clinical AI assistant specialized in diabetic foot ulcer assessment.
Analyze the wound image and respond ONLY with a valid JSON object. No explanation, no markdown, no code fences.

{
  "infection": true or false,
  "ischemia": true or false,
  "severity": 0.0 to 10.0,
  "wound_area_cm2": estimated area as float,
  "description": "one paragraph clinical description in English",
  "confidence": 0.0 to 1.0
}

Assessment criteria:
- infection: Look for erythema, purulent discharge, warmth indicators, tissue necrosis, perilesional inflammation
- ischemia: Look for pallor, cyanosis, lack of granulation tissue, dry necrosis, pale wound bed
- severity: 0=minimal/healing, 5=moderate progression, 10=critical/limb-threatening
- wound_area_cm2: estimate based on proportion of visible foot area (average adult foot ~150 cm2)
- confidence: your confidence in the assessment (0=very uncertain, 1=highly confident)"""

USER_PROMPT = "Analyze this diabetic foot wound image and provide a structured clinical assessment."

print("시스템 프롬프트 설정 완료")

In [ ]:
# ── 4. 데이터셋 경로 확인 ────────────────────────────────────────────────────
DFU_DIR = Path("/kaggle/input/diabetic-foot-ulcer-dfu")
SEG_DIR = Path("/kaggle/input/wound-segmentation-images")

print("=== DFU Binary Dataset ===")
if DFU_DIR.exists():
    for p in sorted(DFU_DIR.iterdir())[:15]:
        count = len(list(p.glob("*.jpg"))) + len(list(p.glob("*.png"))) if p.is_dir() else 0
        print(f"  {p.name}/  ({count} images)")
else:
    print("  [없음] Kaggle에 데이터셋을 추가하세요")

print("\n=== Segmentation Dataset ===")
if SEG_DIR.exists():
    for p in sorted(SEG_DIR.rglob("*"))[:15]:
        print(f"  {p.relative_to(SEG_DIR)}")
else:
    print("  [없음] Kaggle에 데이터셋을 추가하세요")

In [ ]:
# ── 5. JSON 응답 생성 유틸 ────────────────────────────────────────────────────
# ai_service.py의 parse_gemma_response와 동일한 필드 구조

ULCER_KEYWORDS   = {"ulcer", "dfu", "wound", "positive", "diabetic"}
HEALTHY_KEYWORDS = {"healthy", "normal", "non_dfu", "negative", "control"}


def build_json_response(is_ulcer: bool, wound_area_ratio: float = 0.0) -> str:
    """학습용 JSON 응답 생성. ai_service.py parse_gemma_response와 동일한 필드."""
    if not is_ulcer:
        resp = {
            "infection": False,
            "ischemia": False,
            "severity": round(random.uniform(0.5, 1.5), 1),
            "wound_area_cm2": 0.0,
            "description": (
                "No wound detected. Foot skin appears intact with no signs of ulceration, "
                "necrosis, or inflammatory changes. Peripheral tissue looks healthy. "
                "Continue routine monitoring."
            ),
            "confidence": round(random.uniform(0.85, 0.95), 2),
        }
        return json.dumps(resp, indent=2)

    # 면적 비율 → cm² (성인 발 평균 150 cm²)
    area_cm2 = round(max(wound_area_ratio * 150, 0.5), 1)

    if wound_area_ratio < 0.02:       # 소형 (~0–3 cm²)
        infection = random.random() < 0.35
        ischemia  = False
        severity  = round(random.uniform(2.0, 4.0), 1)
        desc = (
            f"Small diabetic foot ulcer observed with limited wound extent ({area_cm2} cm²). "
            "Early-stage lesion with minimal surrounding tissue involvement."
        )
    elif wound_area_ratio < 0.05:     # 중형 (~3–7.5 cm²)
        infection = random.random() < 0.65
        ischemia  = random.random() < 0.30
        severity  = round(random.uniform(4.0, 6.5), 1)
        desc = (
            f"Moderate diabetic foot ulcer ({area_cm2} cm²) with progressive tissue involvement. "
            "Wound margins show moderate erythema."
        )
    else:                              # 대형 (>7.5 cm²)
        infection = True
        ischemia  = random.random() < 0.60
        severity  = round(random.uniform(6.5, 9.5), 1)
        desc = (
            f"Significant diabetic foot ulcer with extensive tissue damage ({area_cm2} cm²). "
            "High risk of systemic complications."
        )

    if infection:
        desc += " Erythema and purulent exudate present, consistent with bacterial colonization."
    if ischemia:
        desc += " Compromised peripheral blood flow noted with pale wound bed and dry necrotic margins."
    if severity >= 7.0:
        desc += " Immediate medical attention strongly recommended to prevent limb loss."
    elif severity >= 5.0:
        desc += " Urgent clinical evaluation advised within 48 hours."
    else:
        desc += " Close outpatient monitoring recommended."

    resp = {
        "infection": infection,
        "ischemia": ischemia,
        "severity": severity,
        "wound_area_cm2": area_cm2,
        "description": desc,
        "confidence": round(random.uniform(0.68, 0.85), 2),
    }
    return json.dumps(resp, indent=2)


def make_conversation(image_path: str, is_ulcer: bool, wound_area_ratio: float = 0.0) -> dict:
    """Gemma 4 멀티모달 chat format. system turn에 SYSTEM_PROMPT 포함."""
    abs_path = str(Path(image_path).resolve())
    return {
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": f"file://{abs_path}"},
                    {"type": "text",  "text": USER_PROMPT},
                ],
            },
            {
                "role": "assistant",
                "content": build_json_response(is_ulcer, wound_area_ratio),
            },
        ]
    }


print("유틸 함수 로드 완료")
# 샘플 확인
print("\n[정상 발 샘플]")
print(json.loads(build_json_response(False))["description"][:80])
print("\n[궤양 샘플 (대형)]")
print(build_json_response(True, 0.08)[:200])

In [ ]:
# ── 6. 데이터 로드 ────────────────────────────────────────────────────────────
samples = []

# ── Dataset 1: laithjj/diabetic-foot-ulcer-dfu (폴더 기반 이진 분류) ──
if DFU_DIR.exists():
    dfu_count = 0
    for folder in DFU_DIR.rglob("*"):
        if not folder.is_dir():
            continue
        name = folder.name.lower()
        if any(k in name for k in ULCER_KEYWORDS):
            label = True
        elif any(k in name for k in HEALTHY_KEYWORDS):
            label = False
        else:
            continue
        for ext in ("*.jpg", "*.jpeg", "*.png"):
            for img in folder.glob(ext):
                samples.append(make_conversation(str(img), is_ulcer=label))
                dfu_count += 1
    print(f"DFU binary: {dfu_count}개")
else:
    print("[SKIP] DFU binary 데이터셋 없음")

# ── Dataset 2: leoscode/wound-segmentation-images (마스크 기반 면적) ──
if SEG_DIR.exists():
    image_dir = next((d for d in SEG_DIR.rglob("images") if d.is_dir()), None)
    mask_dir  = next((d for d in SEG_DIR.rglob("masks")  if d.is_dir()), None)

    if image_dir and mask_dir:
        seg_count = 0
        for img_path in sorted(image_dir.glob("*.jpg")):
            mask_path = mask_dir / f"{img_path.stem}.png"
            if not mask_path.exists():
                mask_path = mask_dir / f"{img_path.stem}.jpg"
            if mask_path.exists():
                mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
                _, binary = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
                ratio = np.count_nonzero(binary) / binary.size
            else:
                ratio = 0.05  # 마스크 없으면 중간값
            samples.append(make_conversation(str(img_path), is_ulcer=True, wound_area_ratio=ratio))
            seg_count += 1
        print(f"Segmentation: {seg_count}개")
    else:
        print("[SKIP] Segmentation images/masks 폴더 구조 확인 필요")
else:
    print("[SKIP] Segmentation 데이터셋 없음")

print(f"\n총 샘플: {len(samples)}개")

if len(samples) == 0:
    print("\n[경고] 샘플이 없습니다. 데이터셋을 Kaggle에 추가했는지 확인하세요.")
    raise SystemExit(0)

In [ ]:
# ── 7. Train / Val 분할 → HuggingFace Dataset ────────────────────────────────
random.shuffle(samples)
split     = int(len(samples) * 0.85)
train_raw = samples[:split]
val_raw   = samples[split:]

train_dataset = Dataset.from_list(train_raw)
val_dataset   = Dataset.from_list(val_raw)

print(f"Train: {len(train_dataset)}개 / Val: {len(val_dataset)}개")
print("\n샘플 확인 (messages 구조):")
sample = train_dataset[0]
for msg in sample["messages"]:
    role = msg["role"]
    content = msg["content"]
    if isinstance(content, list):
        types = [c["type"] for c in content]
        print(f"  [{role}] content types: {types}")
    else:
        print(f"  [{role}] {str(content)[:80]}...")

In [ ]:
# ── 8. 모델 로드 + LoRA 설정 ─────────────────────────────────────────────────
# 주의: 이 셀을 재실행하면 모델을 새로 로드하므로 LoRA 중복 에러 없음
#
# 모델 ID 옵션 (HuggingFace에서 존재 확인 후 선택):
#   unsloth/gemma-4-4b-it    ← 권장 (Unsloth 최적화 버전)
#   google/gemma-4-4b-it     ← 공식 구글 버전

MODEL_ID = "unsloth/gemma-4-4b-it"
MAX_SEQ_LEN = 2048

model, processor = FastVisionModel.from_pretrained(
    MODEL_ID,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state=42,
)

model.print_trainable_parameters()

In [ ]:
# ── 9. Trainer 설정 ───────────────────────────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    tokenizer=processor,
    data_collator=UnslothVisionDataCollator(model, processor),
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,       # effective batch = 8
        num_train_epochs=3,
        learning_rate=2e-4,
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        fp16=True,
        max_seq_length=MAX_SEQ_LEN,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=2,
        output_dir="/kaggle/working/woundwatch-checkpoints",
        report_to="none",
        remove_unused_columns=False,
        dataset_kwargs={"skip_prepare_dataset": True},
    ),
)

print("Trainer 준비 완료")

In [ ]:
# ── 10. 파인튜닝 실행 ─────────────────────────────────────────────────────────
trainer_stats = trainer.train()

print(f"\n훈련 완료")
print(f"총 스텝: {trainer_stats.global_step}")
print(f"최종 Loss: {trainer_stats.training_loss:.4f}")

In [ ]:
# ── 11. 추론 테스트 — JSON 출력 검증 ─────────────────────────────────────────
import json, re

FastVisionModel.for_inference(model)

# 첫 번째 val 샘플로 테스트
test_sample = val_raw[0]
img_uri = test_sample["messages"][1]["content"][0]["image"]  # file:// URI
img_path = img_uri.replace("file://", "")
test_image = Image.open(img_path).convert("RGB")

messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT,
    },
    {
        "role": "user",
        "content": [
            {"type": "image", "image": test_image},
            {"type": "text",  "text": USER_PROMPT},
        ],
    },
]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    do_sample=False,
)
raw_text = processor.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True,
)

print("=== Raw output ===")
print(raw_text)

# JSON 파싱 검증 (ai_service.parse_gemma_response와 동일 로직)
match = re.search(r'\{.*\}', raw_text, re.DOTALL)
if match:
    try:
        parsed = json.loads(match.group())
        required = ["infection", "ischemia", "severity", "wound_area_cm2", "description", "confidence"]
        missing  = [k for k in required if k not in parsed]
        print("\n=== 파싱 성공 ===")
        print(f"infection: {parsed.get('infection')}")
        print(f"ischemia:  {parsed.get('ischemia')}")
        print(f"severity:  {parsed.get('severity')}")
        print(f"area cm²:  {parsed.get('wound_area_cm2')}")
        print(f"confidence:{parsed.get('confidence')}")
        if missing:
            print(f"[경고] 누락 필드: {missing}")
        else:
            print("[OK] 모든 필드 존재")
    except json.JSONDecodeError as e:
        print(f"[실패] JSON 파싱 오류: {e}")
else:
    print("[실패] JSON 블록을 찾을 수 없음 — 추가 학습 필요")

In [ ]:
# ── 12. 모델 저장 ─────────────────────────────────────────────────────────────
SAVE_DIR = "/kaggle/working/woundwatch-gemma4"

# (A) LoRA 어댑터만 저장 (가볍게 HF에 올릴 때 사용)
model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
print(f"LoRA 어댑터 저장: {SAVE_DIR}")

# (B) GGUF 저장 — Ollama 트랙용
# Q4_K_M: 4-bit 양자화, 품질/속도 균형 최적
model.save_pretrained_gguf(
    SAVE_DIR + "-gguf",
    processor,
    quantization_method="q4_k_m",
)
print(f"GGUF 저장: {SAVE_DIR}-gguf/")

import os
gguf_files = list(Path(SAVE_DIR + "-gguf").glob("*.gguf"))
for f in gguf_files:
    size_mb = f.stat().st_size / (1024**2)
    print(f"  {f.name}: {size_mb:.0f} MB")

In [ ]:
# ── 13. Modelfile 생성 (Ollama용) ─────────────────────────────────────────────
# Ollama에서 'ollama create woundwatch -f Modelfile' 로 등록 시 사용

gguf_path = gguf_files[0] if gguf_files else Path(SAVE_DIR + "-gguf/model.gguf")

modelfile_content = f"""FROM {gguf_path.name}

SYSTEM \""""
{SYSTEM_PROMPT}
\""""

PARAMETER temperature 0.1
PARAMETER top_p 0.9
PARAMETER num_predict 256
"""

modelfile_path = Path(SAVE_DIR + "-gguf/Modelfile")
modelfile_path.write_text(modelfile_content)
print(f"Modelfile 저장: {modelfile_path}")
print("\n사용법:")
print(f"  cp {gguf_path} ~/woundwatch/")
print(f"  cp {modelfile_path} ~/woundwatch/")
print("  ollama create woundwatch -f ~/woundwatch/Modelfile")
print("  ollama run woundwatch")

In [ ]:
# ── 14. HuggingFace Hub 업로드 ────────────────────────────────────────────────
# Kaggle Secrets에 HF_TOKEN이 있어야 함 (Write 권한 필요)

from kaggle_secrets import UserSecretsClient

secrets  = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")

# 본인 HuggingFace 사용자명으로 변경
HF_USERNAME = "5seoyoung"   # ← 변경 필요
HF_REPO     = f"{HF_USERNAME}/woundwatch-gemma4-dfu"

model.push_to_hub(HF_REPO, token=hf_token)
processor.push_to_hub(HF_REPO, token=hf_token)

# GGUF도 업로드
model.push_to_hub_gguf(
    HF_REPO,
    processor,
    quantization_method="q4_k_m",
    token=hf_token,
)

print(f"\n업로드 완료!")
print(f"HF 모델: https://huggingface.co/{HF_REPO}")
print(f"Ollama: ollama run hf.co/{HF_REPO}:Q4_K_M")

## 업로드 후 연결 방법

### 방법 A — HuggingFace 직접 로드 (백엔드)
```python
# backend/app/services/ai_service.py
from unsloth import FastVisionModel

model, processor = FastVisionModel.from_pretrained(
    "5seoyoung/woundwatch-gemma4-dfu",
    load_in_4bit=True,
)
FastVisionModel.for_inference(model)
```

### 방법 B — Ollama 로컬 실행
```bash
# HF에서 바로 실행 (GGUF 자동 다운로드)
ollama run hf.co/5seoyoung/woundwatch-gemma4-dfu:Q4_K_M

# 또는 직접 등록
ollama create woundwatch -f Modelfile
ollama run woundwatch
```

### .env.production 업데이트
```
VITE_API_URL=https://5seoyoung-woundwatch.hf.space
```